# 04 · Quantization ladder and benchmark

fp32 → fp16 → int8 (dynamic) → int8 (static) → int4 weight-only, measured on the same images.

**fp4 is not measured here.** It needs Blackwell-class hardware and NVIDIA ModelOpt; there is no ONNX Runtime path that runs on Kaggle. It goes in the report as future work with that reason — a made-up fp4 row is worse than an honest gap.


In [ ]:
# Kaggle setup — run this first in every notebook.
# Settings -> Accelerator: GPU T4 x2 | Internet: ON
import os, subprocess, sys
REPO = "/kaggle/working/SKIT-DS-2023-2027-015"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/shankerjoshi/SKIT-DS-2023-2027-015.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common", "shap"], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


In [ ]:
!python -m src.quantize --ckpt runs/best.pt --calib "data/raw/plantvillage/New Plant Diseases Dataset(Augmented)/valid" --out results/quant


In [ ]:
import pandas as pd
from IPython.display import Markdown
r = pd.read_json("results/quant/quantization_report.json")
r["size_reduction_x"] = (r.size_mb.max() / r.size_mb).round(2)
display(r[["precision", "size_mb", "size_reduction_x", "latency_ms_per_image", "top1_accuracy"]])
print(open("results/quant/quantization_report.md").read())


### What to write in the report
One table, one sentence per rung: what it costs in accuracy and what it saves in size. The deployment pick is almost always int8 static — fp16 buys little on CPU, int4 costs too much accuracy.
